In [7]:
# Import the necessary libraries 
from pyspark.sql import SparkSession
from pathlib import Path
import sqlqueries as sq
import sys
import os
import warnings
import utils.master as ma
from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.functions import sum as spark_sum, when
warnings.filterwarnings("ignore")


#Set the path for logging outputs
job_name = "sales_ETL"
data_base_path = Path("../Logs") # path for logging data
data_working_path = os.path.join(data_base_path, job_name) 
os.makedirs(data_working_path, exist_ok=True)
ma.set_logging_path(data_working_path)

# Spark initialization locally for development
spark = (
    SparkSession.builder
    .appName("sales-bronze-ingestion")
    .getOrCreate()
)

ma.log("Spark Session initialized")

df = spark.read.parquet(
    "../data/bronze/sales"
)
ma.log("Spark DataFrame created from bronze sales parquet files")

2026-02-01 21:26:04: Spark Session initialized
2026-02-01 21:26:05: Spark DataFrame created from bronze sales parquet files


# Heading Data Cleaning
**1. Examine Nulls. Note that data quality validations were implemented using conditional logging, ensuring that  meaningful anomalies (such as null patterns) are recorded.
2. Order_Date was initially set as a string - this column will be used for the partition strategy so it will cast to timestampt
3. Deal with duplicate values --> Duplicates were identified using the composite key (Order_ID, Product), assuming each product appears at most once per order.**

In [8]:
df.createOrReplaceTempView("sales_data")
ma.log("Temporary view 'sales_data' created from sales df")

# Check for nulls and collect results
ma.log("Check for nulls in all rows of sales_data")
df_total_nulls  = spark.sql(sq.total_fully_null_rows)

ma.log("Check for nulls in each column of sales_data")
df_column_nulls  = spark.sql(sq.nulls_per_column)

total_null_rows = df_total_nulls.collect()[0][0]
column_nulls = df_column_nulls.collect()[0].asDict()

# Now log only is something is found
if total_null_rows > 0:
    ma.log(f"DATA QUALITY ISSUE | Fully null rows detected: {total_null_rows}")
else :
    ma.log("No fully null rows detected")

columns_with_nulls = {k: v for k, v in column_nulls.items() if v > 0}

if columns_with_nulls:
    ma.log(f"DATA QUALITY ISSUE | Nulls detected per column: {columns_with_nulls}")
else: 
    ma.log("No nulls detected in any column")

# Raise error if fully null rows exceed 50% of the dataset 
if total_null_rows > df.count() * 0.5:
    raise ValueError("Critical data quality issue: fully null rows detected")

#### A threshold based validation was implemented to fail the pipeline only when fully null rows exceed 50% of the dataset, preventing both false positives and silent catastrophic failures ####


2026-02-01 21:26:06: Temporary view 'sales_data' created from sales df
2026-02-01 21:26:06: Check for nulls in all rows of sales_data
2026-02-01 21:26:06: Check for nulls in each column of sales_data
2026-02-01 21:26:07: No fully null rows detected
2026-02-01 21:26:07: No nulls detected in any column


**The validation showed that the Order_Date column contains no null values. This is a critical result, as Order_Date will be used as the basis for time-based partitioning in downstream layers. Ensuring completeness at this stage guarantees that no records will be excluded or misrouted during partitioning. Given this validation, the column can be safely converted from string to timestamp format in preparation for Silver-layer transformations.**

In [9]:
df = df.withColumn(
    "Order_Date",
    to_timestamp(col("Order_Date"), "MM/dd/yy HH:mm")
)

# Create a data quality check when the date is null to raise an error
invalid_dates = df.select(spark_sum(when(col("Order_Date").isNull(), 1).otherwise(0)).alias("invalid_order_dates")).collect()[0][0]

if invalid_dates  > 0:
    raise ValueError("Critical data quality issue: Order_Date column contains invalid dates")
else:
    ma.log("Order_Date successfully converted to timestamp")

# Order_Date is can later be used to derive partition columns such as year and month without introducing skew or data loss
df = df.withColumn("Order_Year", F.year("Order_Date")).withColumn("Order_Month", F.month("Order_Date"))
ma.log("Order_Year and Order_Month columns created from Order_Date")

# Create a new temporary view with the enriched data
df.createOrReplaceTempView("sales_data_enriched")


2026-02-01 21:26:10: Order_Date successfully converted to timestamp
2026-02-01 21:26:10: Order_Year and Order_Month columns created from Order_Date


Assumption --> One order can contain multiple products 
           --> The same product should not appear multiple times in the same order

Duplicates were identified using the composite key (Order_ID, Product), assuming each product appears at most once per order and considering that the most recent timestamp is the most reliable version. Thus, when duplicates were detected, the most recent record based on Order_Date was retained.


In [10]:
ma.log("Check for duplicate rows in sales_data")
total_duplicates = spark.sql(sq.douplicates_count).collect()[0][0]
ma.log(f"Total duplicate rows detected: {total_duplicates}")

df = spark.sql(sq.remove_duplicates)
ma.log("Duplicate rows removed from sales_data_enriched")
df.createOrReplaceTempView("sales_data_enriched")

2026-02-01 21:26:13: Check for duplicate rows in sales_data


2026-02-01 21:26:14: Total duplicate rows detected: 264
2026-02-01 21:26:14: Duplicate rows removed from sales_data_enriched


In [12]:
# Write Parquet file to silver - cleansed  layer 
ma.log("Start ingestion of Parquet file to cleansed layer")
(
    df.write
    .mode("overwrite")
    .partitionBy("Order_Year", "Order_Month")
    .parquet("../data/cleansed/sales")
)

ma.log("Cleansed sales data written to ../data/cleansed/sales partitioned by year and month")

2026-02-01 21:52:41: Start ingestion of Parquet file to cleansed layer


26/02/01 21:52:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/01 21:52:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/01 21:52:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/01 21:52:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/01 21:52:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/01 21:52:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/02/01 21:52:46 WARN MemoryManager: Total allocation exceeds 95.00% 

2026-02-01 21:53:19: Cleansed sales data written to ../data/cleansed/sales partitioned by year and month
